# 🌱 CSIRO Image2Biomass - Fast Test Version

**Optimized for quick testing (30 minutes)**

⚙️ **CRITICAL Settings**:
- ✅ **Accelerator: GPU T4 x2** ← MUST ENABLE!
- ✅ **Internet: ON**

**This version:**
- 5 epochs (instead of 15)
- 2 folds (instead of 5)
- Smaller model (efficientnet_b0)
- ~30 minutes total

In [ ]:
%%time
!pip install -q timm==0.9.12

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("\n" + "="*60)
    print("❌ ERROR: GPU NOT ENABLED!")
    print("Please enable GPU: Settings → Accelerator → GPU T4 x2")
    print("="*60)
    raise RuntimeError("GPU is required! Please enable it in Settings.")

In [ ]:
import os
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import random
import timm

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# Fast config for testing
CONFIG = {
    'data_dir': '/kaggle/input/csiro-biomass',
    'img_size': 256,  # Smaller for speed
    'batch_size': 32,  # Larger batch
    'num_epochs': 5,   # Quick test
    'lr': 5e-4,
    'n_folds': 2,      # Just 2 folds
    'seed': 42,
    'model_name': 'efficientnet_b0',  # Smaller model
    'device': 'cuda'
}

print("✅ Config loaded (FAST mode)")
print(f"Expected time: ~30 minutes")

In [ ]:
class BiomassDataset(Dataset):
    def __init__(self, df, image_dir, img_size=256, transform=None, is_train=True):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.img_size = img_size
        self.transform = transform
        self.is_train = is_train
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row.get('sample_id', row.get('id', idx))
        img_name = row.get('image_path', f"{img_id}.jpg")
        img_path = os.path.join(self.image_dir, img_name)
        
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (self.img_size, self.img_size), (128, 128, 128))
        
        if self.transform:
            image = self.transform(image)
        
        sample = {'image': image, 'id': img_id}
        if self.is_train and 'target' in row.index:
            sample['target'] = torch.tensor([row['target']], dtype=torch.float32)
        
        return sample

def get_transforms(img_size, is_train=True):
    if is_train:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

class BiomassModel(nn.Module):
    def __init__(self, model_name='efficientnet_b0', pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, 
                                         num_classes=0, global_pool='avg')
        with torch.no_grad():
            feat_dim = self.backbone(torch.randn(1, 3, 224, 224)).shape[1]
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )
    
    def forward(self, x):
        if isinstance(x, dict):
            x = x['image']
        return self.head(self.backbone(x))

print("✅ Dataset and Model defined")

In [ ]:
from tqdm.auto import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = self.avg = self.sum = self.count = 0
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    losses = AverageMeter()
    for batch in tqdm(loader, desc='Train'):
        images = batch['image'].to(device)
        targets = batch['target'].to(device)
        
        preds = model(images)
        loss = criterion(preds, targets)
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        losses.update(loss.item(), images.size(0))
    return losses.avg

def validate(model, loader, criterion, device):
    model.eval()
    preds_list, targets_list = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Valid'):
            images = batch['image'].to(device)
            targets = batch['target'].to(device)
            preds = model(images)
            preds_list.append(preds.cpu().numpy())
            targets_list.append(targets.cpu().numpy())
    preds = np.concatenate(preds_list)
    targets = np.concatenate(targets_list)
    return np.sqrt(np.mean((preds - targets)**2))

def simple_kfold(df, n_splits, seed):
    np.random.seed(seed)
    indices = np.arange(len(df))
    np.random.shuffle(indices)
    fold_size = len(df) // n_splits
    for fold in range(n_splits):
        val_start = fold * fold_size
        val_end = (fold + 1) * fold_size if fold < n_splits - 1 else len(df)
        val_idx = indices[val_start:val_end]
        train_idx = np.concatenate([indices[:val_start], indices[val_end:]])
        yield fold, train_idx, val_idx

print("✅ Training functions ready")

In [ ]:
train_df = pd.read_csv(f"{CONFIG['data_dir']}/train.csv")
print(f"Train: {len(train_df)} samples")
print(f"Target range: [{train_df['target'].min():.1f}, {train_df['target'].max():.1f}]")

In [ ]:
%%time
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
fold_scores = []

for fold, train_idx, valid_idx in simple_kfold(train_df, CONFIG['n_folds'], CONFIG['seed']):
    print(f"\n{'='*50}\nFold {fold}\n{'='*50}")
    
    train_data = train_df.iloc[train_idx].reset_index(drop=True)
    valid_data = train_df.iloc[valid_idx].reset_index(drop=True)
    
    train_loader = DataLoader(
        BiomassDataset(train_data, CONFIG['data_dir'], CONFIG['img_size'], 
                      get_transforms(CONFIG['img_size'], True), True),
        batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True
    )
    valid_loader = DataLoader(
        BiomassDataset(valid_data, CONFIG['data_dir'], CONFIG['img_size'], 
                      get_transforms(CONFIG['img_size'], False), True),
        batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True
    )
    
    model = BiomassModel(CONFIG['model_name'], True).to(CONFIG['device'])
    optimizer = AdamW(model.parameters(), lr=CONFIG['lr'])
    scheduler = CosineAnnealingLR(optimizer, CONFIG['num_epochs'])
    criterion = nn.SmoothL1Loss()
    
    best_rmse = float('inf')
    
    for epoch in range(CONFIG['num_epochs']):
        print(f"\nEpoch {epoch+1}/{CONFIG['num_epochs']}")
        train_loss = train_epoch(model, train_loader, criterion, optimizer, CONFIG['device'])
        valid_rmse = validate(model, valid_loader, criterion, CONFIG['device'])
        scheduler.step()
        
        print(f"Train Loss: {train_loss:.4f}, Valid RMSE: {valid_rmse:.4f}")
        
        if valid_rmse < best_rmse:
            best_rmse = valid_rmse
            torch.save(model.state_dict(), f'/kaggle/working/checkpoints/fold{fold}.pth')
            print(f"✅ Saved! Best RMSE: {best_rmse:.4f}")
    
    fold_scores.append(best_rmse)
    print(f"\nFold {fold} Best: {best_rmse:.4f}")

print(f"\n{'='*50}")
print(f"Mean RMSE: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
print(f"{'='*50}")

## ⚠️ Turn OFF Internet
Settings → Internet → OFF

In [ ]:
test_df = pd.read_csv(f"{CONFIG['data_dir']}/test.csv")
test_loader = DataLoader(
    BiomassDataset(test_df, CONFIG['data_dir'], CONFIG['img_size'], 
                  get_transforms(CONFIG['img_size'], False), False),
    batch_size=64, shuffle=False, num_workers=2, pin_memory=True
)

all_preds = []
for fold in range(CONFIG['n_folds']):
    model = BiomassModel(CONFIG['model_name'], False).to(CONFIG['device'])
    model.load_state_dict(torch.load(f'/kaggle/working/checkpoints/fold{fold}.pth'))
    model.eval()
    
    fold_preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Fold {fold}"):
            preds = model(batch['image'].to(CONFIG['device']))
            fold_preds.append(preds.cpu().numpy())
    all_preds.append(np.concatenate(fold_preds))

final = np.mean(all_preds, axis=0).squeeze()

id_col = 'sample_id' if 'sample_id' in test_df.columns else 'id'
submission = pd.DataFrame({id_col: test_df[id_col], 'target': final})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print("\n✅ Submission ready!")
print(submission.head())
print(f"\nStats: {submission['target'].describe()}")